In [222]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

In [223]:
df = pd.read_csv("/content/drive/MyDrive/INTERN/week3&4/project_resources1/survey_results.csv")

In [224]:
print(len(df))

30010


In [225]:
df.sample(1)

,respondent_id,age,gender,zone,occupation,income_levels,consume_frequency(weekly),current_brand,preferable_consumption_size,awareness_of_other_brands,reasons_for_choosing_brands,flavor_preference,purchase_channel,packaging_preference,health_concerns,typical_consumption_situations,price_range
26899,R26890,30,F,Semi-Urban,Entrepreneur,16L - 25L,3-4 times,Newcomer,Medium (500 ml),2 to 4,Availability,Traditional,Retail Store,Simple,Low (Not very concerned),Social (eg. Parties),100-150


In [226]:
df = df.drop_duplicates(subset=['respondent_id'])

In [227]:
print(len(df))

30000


In [228]:
df = df[df['age'].between(18, 100)]

In [229]:
(df['age'] > 100).sum()

np.int64(0)

In [230]:
bins = [18, 25, 35, 45, 55, 70, float('inf')]
labels = ['18-25', '26-35', '36-45', '46-55', '55-70', '70+']

df['age_group'] = pd.cut(
    df['age'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [231]:
df.sample(5)

,respondent_id,age,gender,zone,occupation,income_levels,consume_frequency(weekly),current_brand,preferable_consumption_size,awareness_of_other_brands,reasons_for_choosing_brands,flavor_preference,purchase_channel,packaging_preference,health_concerns,typical_consumption_situations,price_range,age_group
11280,R11276,29,F,Metro,Working Professional,16L - 25L,5-7 times,Newcomer,Medium (500 ml),above 4,Brand Reputation,Traditional,Online,Eco-Friendly,Low (Not very concerned),"Active (eg. Sports, gym)",200-250,26-35
21598,R21592,50,M,Metro,Working Professional,10L - 15L,3-4 times,Established,Medium (500 ml),above 4,Price,Exotic,Retail Store,Simple,Medium (Moderately health-conscious),Casual (eg. At home),200-250,46-55
13616,R13611,26,F,Metro,Working Professional,<10L,5-7 times,Newcomer,Small (250 ml),0 to 1,Price,Traditional,Retail Store,Simple,Low (Not very concerned),"Active (eg. Sports, gym)",100-150,26-35
10213,R10209,42,M,Semi-Urban,Working Professional,10L - 15L,5-7 times,Established,Medium (500 ml),0 to 1,Price,Traditional,Retail Store,Eco-Friendly,Low (Not very concerned),Social (eg. Parties),100-150,36-45
20583,R20577,23,M,Metro,Student,NaN,3-4 times,Established,Medium (500 ml),0 to 1,Price,Traditional,Online,Simple,High (Very health-conscious),"Active (eg. Sports, gym)",100-150,18-25


In [232]:
pd.crosstab(
    df['age_group'],
    df['occupation']
)

occupation,Entrepreneur,Retired,Student,Working Professional
age_group,,,,
18-25,535,0,7328,2605
26-35,1826,0,697,6570
36-45,1619,0,0,4353
46-55,799,0,0,2167
55-70,221,1130,35,106


In [233]:
print(len(df))

29991


In [270]:
df.isnull().sum()

,0
respondent_id,0
age,0
gender,0
zone,0
occupation,0
income_levels,0
consume_frequency(weekly),8
current_brand,0
preferable_consumption_size,0
awareness_of_other_brands,0


In [234]:
drop_rows = []

for index, row in df.iterrows():
  if row['age_group'] == '55-70' and row['occupation'] == 'Student':
    drop_rows.append(index)

In [235]:
df = df.drop(drop_rows)

In [236]:
print(len(df))

29956


In [237]:
consume_frequency_mapping = {
    "0-2 times" : 1,
    "3-4 times" : 2,
    "5-7 times" : 3
}

awareness_of_other_brands_mapping = {
    "0 to 1" : 1,
    "2 to 4" : 2,
    "above 4" : 3
}

In [238]:
df['consume_frequency(weekly)'] = df['consume_frequency(weekly)'].map(consume_frequency_mapping)

In [239]:
df['awareness_of_other_brands'] = df['awareness_of_other_brands'].map(awareness_of_other_brands_mapping)

In [240]:
df.sample(2)

,respondent_id,age,gender,zone,occupation,income_levels,consume_frequency(weekly),current_brand,preferable_consumption_size,awareness_of_other_brands,reasons_for_choosing_brands,flavor_preference,purchase_channel,packaging_preference,health_concerns,typical_consumption_situations,price_range,age_group
21526,R21520,25,M,Metro,Working Professional,<10L,2.0,Newcomer,Medium (500 ml),3,Availability,Exotic,Online,Simple,High (Very health-conscious),"Active (eg. Sports, gym)",150-200,18-25
19938,R19932,19,M,Metro,Working Professional,10L - 15L,1.0,Established,Small (250 ml),2,Availability,Exotic,Online,Eco-Friendly,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",100-150,18-25


In [241]:
df['denom'] = (df['awareness_of_other_brands'] + df['consume_frequency(weekly)'])

In [242]:
df['cf_ab_score'] = (df['consume_frequency(weekly)']/df['denom'])

In [243]:
df['cf_ab_score'] = df['cf_ab_score'].round(2)

In [244]:
df['cf_ab_score'].max()

0.75

In [245]:
df.sample(1)

,respondent_id,age,gender,zone,occupation,income_levels,consume_frequency(weekly),current_brand,preferable_consumption_size,awareness_of_other_brands,reasons_for_choosing_brands,flavor_preference,purchase_channel,packaging_preference,health_concerns,typical_consumption_situations,price_range,age_group,denom,cf_ab_score
16700,R16695,50,M,Urban,Entrepreneur,> 35L,1.0,Newcomer,Large (1 L),1,Quality,Exotic,Retail Store,Premium,Low (Not very concerned),Social (eg. Parties),200-250,46-55,2.0,0.5


In [246]:
zone_mapping = {
    "Urban" : 3,
    "Metro" : 4,
    "Rural" : 1,
    "Semi-Urban" : 2
}

income_mapping = {
    "<10L" : 1,
    "10L - 15L" : 2,
    "16L - 25L" : 3,
    "26L - 35L" : 4,
    "> 35L" : 5,
    "Not Reported": 0
}

In [247]:
df['zone'].unique()

array(['Urban', 'Metro', 'Rural', 'Semi-Urban', 'Metor', 'urbna'],
      dtype=object)

In [248]:
df['zone'] = df['zone'].replace({
    'urbna': 'Urban',
    'Metor': 'Metro'
})

In [249]:
print(df['income_levels'].unique())

['<10L' '> 35L' '16L - 25L' nan '10L - 15L' '26L - 35L']


In [250]:
df['income_levels'] = df['income_levels'].fillna('Not Reported')

income_mapping = {
    "<10L": 1,
    "10L - 15L": 2,
    "16L - 25L": 3,
    "26L - 35L": 4,
    "> 35L": 5,
    "Not Reported": 0
}

df['income_score'] = df['income_levels'].map(income_mapping)

In [251]:
df['zone_score'] = df['zone'].map(zone_mapping)
df['income_score'] = df['income_levels'].map(income_mapping)

df['zas_score'] = df['zone_score'] * df['income_score']

print(df['zas_score'].nunique())
print(sorted(df['zas_score'].dropna().unique()))

14
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(8), np.int64(9), np.int64(10), np.int64(12), np.int64(15), np.int64(16), np.int64(20)]


In [252]:
print(df['income_score'].isna().sum())
print(df['zas_score'].isna().sum())

0
0


In [253]:
print(df['income_levels'].unique())

['<10L' '> 35L' '16L - 25L' 'Not Reported' '10L - 15L' '26L - 35L']


In [254]:
df.shape

(29956, 23)

In [255]:
df['age'].describe()

,age
count,29956.000000
mean,32.911570
std,11.865857
min,18.000000
25%,23.000000
50%,31.000000
75%,40.000000
max,70.000000


In [256]:
df['age_group'].describe()

,age_group
count,29956
unique,5
top,18-25
freq,10468


In [257]:
df[['age_group', 'occupation']]

,age_group,occupation
0,26-35,Working Professional
1,46-55,Working Professional
2,36-45,Working Professional
3,26-35,Working Professional
4,18-25,Student
...,...,...
30005,26-35,Working Professional
30006,36-45,Working Professional
30007,55-70,Retired
30008,18-25,Working Professional


In [258]:
pd.crosstab(df['age_group'], df['occupation'])

occupation,Entrepreneur,Retired,Student,Working Professional
age_group,,,,
18-25,535,0,7328,2605
26-35,1826,0,697,6570
36-45,1619,0,0,4353
46-55,799,0,0,2167
55-70,221,1130,0,106


In [259]:
df.sample(1)

,respondent_id,age,gender,zone,occupation,income_levels,consume_frequency(weekly),current_brand,preferable_consumption_size,awareness_of_other_brands,reasons_for_choosing_brands,flavor_preference,purchase_channel,packaging_preference,health_concerns,typical_consumption_situations,price_range,age_group,denom,cf_ab_score,income_score,zone_score,zas_score
25527,R25518,19,F,Semi-Urban,Student,Not Reported,3.0,Established,Small (250 ml),3,Price,Exotic,Online,Premium,Low (Not very concerned),Casual (eg. At home),100-150,18-25,6.0,0.5,0,2,0


In [260]:
df['current_brand'].unique()

array(['Newcomer', 'Established', 'newcomer', 'Establishd'], dtype=object)

In [261]:
df['reasons_for_choosing_brands'].unique()

array(['Price', 'Quality', 'Availability', 'Brand Reputation'],
      dtype=object)

In [262]:
(df['current_brand'] == 'Establishd').sum()

np.int64(20)

In [263]:
df['current_brand'] = df['current_brand'].replace('Establishd', 'Established')

In [264]:
df['current_brand'] = df['current_brand'].replace('newcomer', 'Newcomer')

In [265]:
df['current_brand'].unique()

array(['Newcomer', 'Established'], dtype=object)

In [266]:
# df['bsi'] = 0

# for index, row in df.iterrows():
#   if row['current_brand'] != 'Established' and row['reasons_for_choosing_brands'] in ['Price', 'Quality']:
#     df.at[index, 'bsi'] = 1

In [268]:
df['bsi'] = np.where(
    (df['current_brand'] != 'Established') & (df['reasons_for_choosing_brands'].isin(['Price', 'Quality'])),
    1,
    0
)

In [269]:
df['bsi'].value_counts()

,count
bsi,
0,20796
1,9160
